In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import rankdata, norm
import gc

print("Memulai...")

# ============================================================
# 1. LOKASI FILE
# ============================================================

FILE = Path("Siklon_Tropis_ERA5_Extracted.xlsx")

if not FILE.exists():
    raise FileNotFoundError(
        f"File tidak ditemukan:\n{FILE.resolve()}\n\n"
        "Pastikan file Excel berada di folder kerja Jupyter."
    )

print("File ditemukan:")
print(FILE.resolve())


# ============================================================
# 2. BACA EXCEL
# ============================================================

df = pd.read_excel(FILE)

print("\nData berhasil dibaca.")
print("Jumlah baris :", len(df))
print("Jumlah kolom :", len(df.columns))


# ============================================================
# 3. CEK KOLOM
# ============================================================

print("\nKolom yang tersedia:")

for col in df.columns:
    print(" -", col)


# ============================================================
# 4. CEK KOLOM WAJIB
# ============================================================

required = [
    "SID",
    "ISO_TIME",
    "LAT",
    "LON",
    "era5_sst",
    "era5_tp",
    "era5_w",
    "era5_vo",
    "WIND_km/h",
    "PRESSURE_hPa"
]

missing_columns = [
    col for col in required
    if col not in df.columns
]

if missing_columns:

    raise ValueError(
        "\nKolom berikut tidak ditemukan:\n"
        + "\n".join(missing_columns)
    )

print("\nSemua kolom yang diperlukan tersedia.")


# ============================================================
# 5. KONVERSI WAKTU
# ============================================================

df["ISO_TIME"] = pd.to_datetime(
    df["ISO_TIME"],
    errors="coerce"
)

jumlah_waktu_invalid = df["ISO_TIME"].isna().sum()

print(
    "\nWaktu tidak terbaca :",
    jumlah_waktu_invalid
)


# ============================================================
# 6. BUANG BARIS TANPA SID / WAKTU
# ============================================================

df = df.dropna(
    subset=["SID", "ISO_TIME"]
).copy()

print(
    "Baris setelah pembersihan :",
    len(df)
)


# ============================================================
# 7. URUTKAN BERDASARKAN SID DAN WAKTU
# ============================================================

df = df.sort_values(
    ["SID", "ISO_TIME"]
)


# ============================================================
# 8. AMBIL SATU TITIK GENESIS PER SID
# ============================================================

# Titik pertama secara waktu untuk setiap SID

idx_genesis = (
    df.groupby("SID")["ISO_TIME"]
      .idxmin()
)

genesis = (
    df.loc[idx_genesis]
      .copy()
      .sort_values("ISO_TIME")
      .reset_index(drop=True)
)


# ============================================================
# 9. HASIL AWAL
# ============================================================

print("\n" + "=" * 60)
print("HASIL PEMBENTUKAN GENESIS")
print("=" * 60)

print(
    "Jumlah SID unik dalam data awal :",
    df["SID"].nunique()
)

print(
    "Jumlah titik genesis :",
    len(genesis)
)


# ============================================================
# 10. BEBASKAN MEMORI
# ============================================================

del idx_genesis
gc.collect()

print("\nCELL 1 SELESAI.")

Memulai...
File ditemukan:
C:\Users\LENOVO\Documents\Tugaslah Pokoknya\Draft\Siklon_Tropis_ERA5_Extracted.xlsx

Data berhasil dibaca.
Jumlah baris : 26620
Jumlah kolom : 18

Kolom yang tersedia:
 - SID
 - NAME
 - ISO_TIME
 - LAT
 - LON
 - SEASON
 - BASIN
 - NATURE
 - TYPE
 - WIND_km/h
 - PRESSURE_hPa
 - DIST2LAND
 - LANDFALL
 - era5_tp
 - era5_msl
 - era5_sst
 - era5_w
 - era5_vo

Semua kolom yang diperlukan tersedia.

Waktu tidak terbaca : 0
Baris setelah pembersihan : 26620

HASIL PEMBENTUKAN GENESIS
Jumlah SID unik dalam data awal : 521
Jumlah titik genesis : 521

CELL 1 SELESAI.


In [2]:
# ============================================================
# CELL 2
# SELEKSI DOMAIN PENELITIAN
# ============================================================

print("Memulai seleksi domain...")


# ============================================================
# 1. SALIN DATA GENESIS
# ============================================================

genesis_domain = genesis.copy()


# ============================================================
# 2. DOMAIN PENELITIAN
# ============================================================

genesis_domain = genesis_domain[
    (genesis_domain["LON"] >= 95) &
    (genesis_domain["LON"] <= 145) &
    (genesis_domain["LAT"] >= -15) &
    (genesis_domain["LAT"] <= 25)
].copy()


# ============================================================
# 3. RESET INDEX
# ============================================================

genesis_domain = (
    genesis_domain
    .reset_index(drop=True)
)


# ============================================================
# 4. KLASIFIKASI ZONA
# ============================================================

def klasifikasi_zona(lat):

    if -5 <= lat <= 5:
        return "Ekuatorial"

    elif lat > 5:
        return "Utara"

    else:
        return "Selatan"


genesis_domain["ZONE"] = (
    genesis_domain["LAT"]
    .apply(klasifikasi_zona)
)


# ============================================================
# 5. HASIL
# ============================================================

print("\n" + "=" * 60)
print("HASIL SELEKSI DOMAIN")
print("=" * 60)

print(
    "Jumlah genesis dalam domain :",
    len(genesis_domain)
)


print("\nDistribusi zona:")

print(
    genesis_domain["ZONE"]
    .value_counts()
)


# ============================================================
# 6. CEK VAMEI DAN SENYAR
# ============================================================

target = genesis_domain[
    genesis_domain["NAME"]
    .astype(str)
    .str.upper()
    .isin(["VAMEI", "SENYAR"])
].copy()


print("\n" + "=" * 60)
print("VAMEI DAN SENYAR")
print("=" * 60)

print(
    target[
        [
            "SID",
            "NAME",
            "SEASON",
            "LAT",
            "LON",
            "ZONE"
        ]
    ].to_string(index=False)
)


# ============================================================
# 7. VALIDASI JUMLAH
# ============================================================

if len(genesis_domain) == 392:

    print(
        "\n✓ BENAR: dataset penelitian = 392 siklon."
    )

else:

    print(
        "\n⚠ PERHATIAN!"
    )

    print(
        "Jumlah yang diperoleh:",
        len(genesis_domain)
    )

    print(
        "Kita JANGAN lanjut ke model sebelum "
        "penyebab perbedaan ini diperiksa."
    )


# ============================================================
# 8. VALIDASI KOMPOSISI ZONA
# ============================================================

jumlah_utara = (
    genesis_domain["ZONE"] == "Utara"
).sum()

jumlah_ekuator = (
    genesis_domain["ZONE"] == "Ekuatorial"
).sum()

jumlah_selatan = (
    genesis_domain["ZONE"] == "Selatan"
).sum()


print("\nKomposisi:")
print("Utara      :", jumlah_utara)
print("Ekuatorial :", jumlah_ekuator)
print("Selatan    :", jumlah_selatan)

print(
    "\nTotal       :",
    jumlah_utara
    + jumlah_ekuator
    + jumlah_selatan
)


print("\nCELL 2 SELESAI.")

Memulai seleksi domain...

HASIL SELEKSI DOMAIN
Jumlah genesis dalam domain : 392

Distribusi zona:
ZONE
Utara         354
Selatan        36
Ekuatorial      2
Name: count, dtype: int64

VAMEI DAN SENYAR
          SID   NAME  SEASON  LAT   LON       ZONE
2001361N01106  VAMEI    2001  1.4 106.2 Ekuatorial
2025330N05099 SENYAR    2025  4.8  99.0 Ekuatorial

✓ BENAR: dataset penelitian = 392 siklon.

Komposisi:
Utara      : 354
Ekuatorial : 2
Selatan    : 36

Total       : 392

CELL 2 SELESAI.


In [3]:
# ============================================================
# CELL 3A
# MEMBENTUK DATASET MODEL
# ============================================================

print("Memulai Cell 3A...")

# Salin dataset genesis
genesis_model = genesis_domain.copy()

# ------------------------------------------------------------
# Konversi satuan
# ------------------------------------------------------------

# SST: Kelvin -> Celsius
genesis_model["SST_C"] = (
    genesis_model["era5_sst"] - 273.15
)

# Total precipitation: meter -> mm
genesis_model["TP_mm"] = (
    genesis_model["era5_tp"] * 1000
)

# Relative vorticity: s^-1 -> x10^-5 s^-1
genesis_model["VORT_x1e5"] = (
    genesis_model["era5_vo"] * 1e5
)


# ------------------------------------------------------------
# Bentuk 7 variabel model
# ------------------------------------------------------------

X = pd.DataFrame({
    "Latitude":
        pd.to_numeric(
            genesis_model["LAT"],
            errors="coerce"
        ),

    "SST":
        pd.to_numeric(
            genesis_model["SST_C"],
            errors="coerce"
        ),

    "Total_Precipitation":
        pd.to_numeric(
            genesis_model["TP_mm"],
            errors="coerce"
        ),

    "Vertical_Velocity":
        pd.to_numeric(
            genesis_model["era5_w"],
            errors="coerce"
        ),

    "Relative_Vorticity":
        pd.to_numeric(
            genesis_model["VORT_x1e5"],
            errors="coerce"
        ),

    "Wind":
        pd.to_numeric(
            genesis_model["WIND_km/h"],
            errors="coerce"
        ),

    "Pressure":
        pd.to_numeric(
            genesis_model["PRESSURE_hPa"],
            errors="coerce"
        )
})


print("=" * 60)
print("HASIL CELL 3A")
print("=" * 60)

print("Jumlah sampel :", len(X))
print("Jumlah variabel :", len(X.columns))
print("Ukuran X :", X.shape)

print("\nNama variabel:")
print(X.columns.tolist())

print("\n5 baris pertama:")
print(X.head())

print("\nMissing value:")
print(X.isna().sum())

print("\nCELL 3A SELESAI.")

Memulai Cell 3A...
HASIL CELL 3A
Jumlah sampel : 392
Jumlah variabel : 7
Ukuran X : (392, 7)

Nama variabel:
['Latitude', 'SST', 'Total_Precipitation', 'Vertical_Velocity', 'Relative_Vorticity', 'Wind', 'Pressure']

5 baris pertama:
   Latitude        SST  Total_Precipitation  Vertical_Velocity  \
0      11.9  29.967676             6.782532          -0.861137   
1      11.7  29.688135             1.443386          -0.008365   
2      15.7  28.910791             0.636101          -0.026876   
3       7.3  29.137354             0.946522          -0.028187   
4      13.2  29.337549             0.796318          -0.131378   

   Relative_Vorticity  Wind  Pressure  
0           13.596937  45.0    1002.0  
1            7.426878  45.0    1002.0  
2           11.223648  45.0    1002.0  
3           14.697504  45.0    1002.0  
4            7.541873  45.0    1004.0  

Missing value:
Latitude               0
SST                    6
Total_Precipitation    0
Vertical_Velocity      0
Relative_Vorti

In [4]:
# ============================================================
# CELL 3B
# STATISTIK DESKRIPTIF
# ============================================================

print("Memulai Cell 3B...")

stats = pd.DataFrame({
    "N": X.count(),
    "Mean": X.mean(),
    "Median": X.median(),
    "Std": X.std(),
    "Min": X.min(),
    "Max": X.max(),
    "Skewness": X.skew()
})

print("=" * 60)
print("STATISTIK DESKRIPTIF")
print("=" * 60)

print(stats.round(4))

print("\nCELL 3B SELESAI.")

Memulai Cell 3B...
STATISTIK DESKRIPTIF
                       N       Mean     Median      Std       Min        Max  \
Latitude             392    12.2862    14.2500   8.3636  -14.7000    25.0000   
SST                  386    29.3990    29.3505   0.6448   27.0192    31.2260   
Total_Precipitation  392     2.2534     1.4262   2.6534    0.0010    20.0443   
Vertical_Velocity    392    -0.2999    -0.1698   0.5623   -3.7981     0.9093   
Relative_Vorticity   392    13.1272    12.9917  14.7116  -77.1288    77.8049   
Wind                 391    47.1458    45.0000   5.5472   44.0000    95.0000   
Pressure             392  1002.2526  1003.0000   2.8302  984.0000  1009.0000   

                     Skewness  
Latitude              -1.6879  
SST                   -0.2518  
Total_Precipitation    2.8269  
Vertical_Velocity     -2.4147  
Relative_Vorticity    -0.7239  
Wind                   3.8485  
Pressure              -1.8975  

CELL 3B SELESAI.


In [5]:
# ============================================================
# CELL 3C
# KORELASI
# ============================================================

print("Memulai Cell 3C...")


# ------------------------------------------------------------
# Pearson
# ------------------------------------------------------------

pearson_corr = X.corr(
    method="pearson"
)

print("=" * 60)
print("KORELASI PEARSON")
print("=" * 60)

print(
    pearson_corr.round(3)
)


# ------------------------------------------------------------
# Spearman
# ------------------------------------------------------------

spearman_corr = X.corr(
    method="spearman"
)

print("\n" + "=" * 60)
print("KORELASI SPEARMAN")
print("=" * 60)

print(
    spearman_corr.round(3)
)


print("\nCELL 3C SELESAI.")

Memulai Cell 3C...
KORELASI PEARSON
                     Latitude    SST  Total_Precipitation  Vertical_Velocity  \
Latitude                1.000  0.101               -0.091              0.039   
SST                     0.101  1.000               -0.024              0.091   
Total_Precipitation    -0.091 -0.024                1.000             -0.617   
Vertical_Velocity       0.039  0.091               -0.617              1.000   
Relative_Vorticity      0.611 -0.067                0.195             -0.188   
Wind                    0.097 -0.125                0.164             -0.157   
Pressure               -0.162  0.040               -0.008              0.058   

                     Relative_Vorticity   Wind  Pressure  
Latitude                          0.611  0.097    -0.162  
SST                              -0.067 -0.125     0.040  
Total_Precipitation               0.195  0.164    -0.008  
Vertical_Velocity                -0.188 -0.157     0.058  
Relative_Vorticity          

In [7]:
# ============================================================
# CELL 3D
# TRANSFORMASI EMPIRICAL CDF
# ============================================================

print("Memulai Cell 3D...")

n = len(X)

U = pd.DataFrame(
    index=X.index
)

for col in X.columns:

    # Ranking menggunakan pandas
    ranks = X[col].rank(
        method="average"
    )

    # Mid-rank empirical CDF
    U[col] = (
        ranks - 0.5
    ) / n


print("=" * 60)
print("HASIL ECDF")
print("=" * 60)

print("Ukuran U :", U.shape)

print("\nNilai minimum setiap variabel:")
print(U.min())

print("\nNilai maksimum setiap variabel:")
print(U.max())

print("\n5 baris pertama:")
print(U.head())

print("\nCELL 3D SELESAI.")

Memulai Cell 3D...
HASIL ECDF
Ukuran U : (392, 7)

Nilai minimum setiap variabel:
Latitude               0.001276
SST                    0.001276
Total_Precipitation    0.001276
Vertical_Velocity      0.001276
Relative_Vorticity     0.001276
Wind                   0.001276
Pressure               0.001276
dtype: float64

Nilai maksimum setiap variabel:
Latitude               0.998724
SST                    0.983418
Total_Precipitation    0.998724
Vertical_Velocity      0.998724
Relative_Vorticity     0.998724
Wind                   0.996173
Pressure               0.998724
dtype: float64

5 baris pertama:
   Latitude       SST  Total_Precipitation  Vertical_Velocity  \
0  0.341837  0.802296             0.934949           0.118622   
1  0.323980  0.672194             0.506378           0.707908   
2  0.627551  0.204082             0.297194           0.690051   
3  0.136480  0.294643             0.390306           0.684949   
4  0.420918  0.470663             0.338010           0.542092   

In [8]:
# ============================================================
# CELL 3E
# TRANSFORMASI ECDF -> DISTRIBUSI NORMAL STANDAR
# ============================================================

from statistics import NormalDist

print("Memulai Cell 3E...")

normal_dist = NormalDist()

Z = pd.DataFrame(
    index=U.index
)

for col in U.columns:

    Z[col] = U[col].apply(
        normal_dist.inv_cdf
    )


print("=" * 60)
print("HASIL TRANSFORMASI GAUSSIAN")
print("=" * 60)

print("Ukuran Z :", Z.shape)

print("\nRata-rata:")
print(Z.mean().round(4))

print("\nStandar deviasi:")
print(Z.std().round(4))

print("\n5 baris pertama:")
print(Z.head())

print("\nCELL 3E SELESAI.")

Memulai Cell 3E...
HASIL TRANSFORMASI GAUSSIAN
Ukuran Z : (392, 7)

Rata-rata:
Latitude              -0.0000
SST                   -0.0389
Total_Precipitation    0.0000
Vertical_Velocity     -0.0000
Relative_Vorticity    -0.0000
Wind                   0.0569
Pressure              -0.0214
dtype: float64

Standar deviasi:
Latitude               0.9996
SST                    0.9563
Total_Precipitation    0.9996
Vertical_Velocity      0.9996
Relative_Vorticity     0.9996
Wind                   0.6516
Pressure               0.9390
dtype: float64

5 baris pertama:
   Latitude       SST  Total_Precipitation  Vertical_Velocity  \
0 -0.407456  0.849851             1.513700          -1.181901   
1 -0.456599  0.445979             0.015987           0.547284   
2  0.325374 -0.827130            -0.532488           0.495995   
3 -1.096273 -0.539871            -0.278521           0.481583   
4 -0.199545 -0.073603            -0.417900           0.105705   

   Relative_Vorticity     Wind  Pressure  
0

In [9]:
# ============================================================
# CELL 3F-REVISI
# GAUSSIAN COPULA DENGAN DATA LENGKAP
# ============================================================

print("=" * 70)
print("PEMBENTUKAN ULANG GAUSSIAN COPULA")
print("=" * 70)

# ------------------------------------------------------------
# 1. Ambil hanya baris yang lengkap untuk semua variabel
# ------------------------------------------------------------

X_complete = X.dropna().copy()

print("Jumlah seluruh kejadian :", len(X))
print("Jumlah data lengkap     :", len(X_complete))
print("Data yang dikeluarkan   :", len(X) - len(X_complete))


# ------------------------------------------------------------
# 2. Tampilkan kejadian yang dikeluarkan
# ------------------------------------------------------------

excluded_index = X.index.difference(
    X_complete.index
)

print("\nJumlah kejadian yang tidak lengkap:")
print(len(excluded_index))


if len(excluded_index) > 0:

    print("\nKejadian yang dikeluarkan:")

    print(
        genesis_model.loc[
            excluded_index,
            [
                "SID",
                "NAME",
                "SEASON",
                "LAT",
                "LON",
                "ZONE"
            ]
        ].to_string(index=False)
    )


# ------------------------------------------------------------
# 3. ECDF
# ------------------------------------------------------------

n = len(X_complete)

U_complete = pd.DataFrame(
    index=X_complete.index
)

for col in X_complete.columns:

    ranks = X_complete[col].rank(
        method="average"
    )

    U_complete[col] = (
        ranks - 0.5
    ) / n


# ------------------------------------------------------------
# 4. Transformasi Gaussian
# ------------------------------------------------------------

from statistics import NormalDist

normal_dist = NormalDist()

Z_complete = pd.DataFrame(
    index=U_complete.index
)

for col in U_complete.columns:

    Z_complete[col] = [
        normal_dist.inv_cdf(value)
        for value in U_complete[col]
    ]


# ------------------------------------------------------------
# 5. Matriks Gaussian Copula
# ------------------------------------------------------------

copula_corr = Z_complete.corr(
    method="pearson"
)


print("\n" + "=" * 70)
print("MATRIKS KORELASI GAUSSIAN COPULA")
print("=" * 70)

print(
    copula_corr.round(4)
)


# ------------------------------------------------------------
# 6. Cek data kosong
# ------------------------------------------------------------

print("\nJumlah nilai kosong:")
print(
    copula_corr.isna().sum().sum()
)


# ------------------------------------------------------------
# 7. Cholesky sederhana
# ------------------------------------------------------------

def cholesky_python(matrix):

    n = len(matrix)

    L = [
        [0.0 for _ in range(n)]
        for _ in range(n)
    ]

    for i in range(n):

        for j in range(i + 1):

            total = 0.0

            for k in range(j):

                total += (
                    L[i][k] *
                    L[j][k]
                )

            if i == j:

                value = (
                    matrix[i][i]
                    - total
                )

                if value <= 0:

                    return False, None

                L[i][j] = value ** 0.5

            else:

                if L[j][j] == 0:

                    return False, None

                L[i][j] = (
                    matrix[i][j]
                    - total
                ) / L[j][j]

    return True, L


is_pd, L = cholesky_python(
    copula_corr.values.tolist()
)


print("\n" + "=" * 70)
print("PEMERIKSAAN MATRKS")
print("=" * 70)

if is_pd:

    print(
        "✓ Matriks Gaussian Copula "
        "positive definite."
    )

else:

    print(
        "⚠ Matriks Gaussian Copula "
        "tidak positive definite."
    )


# ------------------------------------------------------------
# 8. Pastikan VAMEI dan SENYAR tetap masuk
# ------------------------------------------------------------

target = genesis_model[
    genesis_model["NAME"]
    .astype(str)
    .str.upper()
    .isin(["VAMEI", "SENYAR"])
]

print("\n" + "=" * 70)
print("VAMEI DAN SENYAR")
print("=" * 70)

print(
    target[
        [
            "SID",
            "NAME",
            "SEASON",
            "LAT",
            "LON",
            "ZONE"
        ]
    ].to_string(index=False)
)

print("\nCELL 3F-REVISI SELESAI.")

PEMBENTUKAN ULANG GAUSSIAN COPULA
Jumlah seluruh kejadian : 392
Jumlah data lengkap     : 385
Data yang dikeluarkan   : 7

Jumlah kejadian yang tidak lengkap:
7

Kejadian yang dikeluarkan:
          SID      NAME  SEASON   LAT   LON    ZONE
2004255N23120     HAIMA    2004  22.8 120.2   Utara
2011021S16129    BIANCA    2011 -14.7 126.4 Selatan
2011045S14132    CARLOS    2011 -13.1 130.8 Selatan
2013178N09133    RUMBIA    2013   9.1 132.5   Utara
2017321N07124    KIROGI    2017   9.9 118.7   Utara
2020160N10133      NURI    2020  15.9 120.6   Utara
2024198N08130 PRAPIROON    2024  12.8 121.0   Utara

MATRIKS KORELASI GAUSSIAN COPULA
                     Latitude     SST  Total_Precipitation  Vertical_Velocity  \
Latitude               1.0000  0.1103              -0.1171             0.0477   
SST                    0.1103  1.0000               0.0288             0.0001   
Total_Precipitation   -0.1171  0.0288               1.0000            -0.4014   
Vertical_Velocity      0.0477  0.0001

In [10]:
# ============================================================
# CELL 3F
# MATRIKS KORELASI GAUSSIAN COPULA
# VERSI AMAN — TANPA EIGENVALUE
# ============================================================

print("Memulai Cell 3F...")


# ------------------------------------------------------------
# 1. HITUNG KORELASI PADA RUANG GAUSSIAN
# ------------------------------------------------------------

copula_corr = Z.corr(
    method="pearson"
)


# ------------------------------------------------------------
# 2. TAMPILKAN HASIL
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MATRIKS KORELASI GAUSSIAN COPULA")
print("=" * 60)

print(
    copula_corr.round(4)
)


# ------------------------------------------------------------
# 3. CEK UKURAN
# ------------------------------------------------------------

print("\nUkuran matriks:")
print(copula_corr.shape)


# ------------------------------------------------------------
# 4. CEK DIAGONAL
# ------------------------------------------------------------

print("\nNilai diagonal:")

print(
    copula_corr.values[
        range(len(copula_corr)),
        range(len(copula_corr))
    ]
)


# ------------------------------------------------------------
# 5. CEK SIMETRI
# ------------------------------------------------------------

is_symmetric = (
    copula_corr
    .equals(copula_corr.T)
)

print(
    "\nMatriks simetris:",
    is_symmetric
)


# ------------------------------------------------------------
# 6. CEK NILAI KOSONG
# ------------------------------------------------------------

print(
    "\nJumlah nilai kosong:",
    copula_corr.isna().sum().sum()
)


print("\nCELL 3F SELESAI.")

Memulai Cell 3F...

MATRIKS KORELASI GAUSSIAN COPULA
                     Latitude     SST  Total_Precipitation  Vertical_Velocity  \
Latitude               1.0000  0.1130              -0.1368             0.0723   
SST                    0.1130  1.0000               0.0265             0.0040   
Total_Precipitation   -0.1368  0.0265               1.0000            -0.4087   
Vertical_Velocity      0.0723  0.0040              -0.4087             1.0000   
Relative_Vorticity     0.3960 -0.0476               0.1323            -0.0958   
Wind                   0.0791 -0.0951               0.1350            -0.0435   
Pressure              -0.1602  0.0310               0.0665            -0.0019   

                     Relative_Vorticity    Wind  Pressure  
Latitude                         0.3960  0.0791   -0.1602  
SST                             -0.0476 -0.0951    0.0310  
Total_Precipitation              0.1323  0.1350    0.0665  
Vertical_Velocity               -0.0958 -0.0435   -0.0019 

In [12]:
# ============================================================
# CELL 3G
# PERSIAPAN SUBSET SIMULATION
# ============================================================

import numpy as np
import math
from statistics import NormalDist

print("Menyiapkan Subset Simulation...")


# ============================================================
# 1. KONFIGURASI
# ============================================================

N = 20000

p0 = 0.10

sigma = 1.0

random_seed = 20261002

rng = np.random.default_rng(
    random_seed
)

normal_dist = NormalDist()


# ============================================================
# 2. CHOLESKY MATRIX
# ============================================================

is_pd, L_list = cholesky_python(
    copula_corr.values.tolist()
)

if not is_pd:

    raise ValueError(
        "Matriks Gaussian Copula tidak positive definite."
    )

L = np.array(
    L_list,
    dtype=float
)


# ============================================================
# 3. SORTED MARGINAL
# ============================================================

sorted_marginals = {}

for col in X_complete.columns:

    sorted_marginals[col] = np.sort(
        X_complete[col].to_numpy()
    )


# ============================================================
# 4. FUNGSI NORMAL CDF
# ============================================================

def normal_cdf(z):

    return 0.5 * (
        1.0 +
        math.erf(
            z / math.sqrt(2.0)
        )
    )


# ============================================================
# 5. EMPIRICAL QUANTILE
# ============================================================

def empirical_ppf(
    u,
    sorted_values
):

    n = len(sorted_values)

    index = math.ceil(
        u * n
    ) - 1

    if index < 0:

        index = 0

    if index >= n:

        index = n - 1

    return float(
        sorted_values[index]
    )


# ============================================================
# 6. TRANSFORMASI LATITUDE
# ============================================================

lat_sorted = sorted_marginals[
    "Latitude"
]


def latitude_from_z(
    z_lat
):

    u = normal_cdf(
        z_lat
    )

    return empirical_ppf(
        u,
        lat_sorted
    )


# ============================================================
# 7. FUNGSI EVENT
# ============================================================

def performance_function(
    y
):

    # y = independent standard normal space

    z = L.dot(y)

    latitude = latitude_from_z(
        z[0]
    )

    return abs(latitude) - 5.0


# ============================================================
# 8. CEK FUNGSI
# ============================================================

print("\nJumlah variabel :", len(X_complete.columns))

print(
    "Jumlah sampel   :", N
)

print(
    "p0              :", p0
)

print(
    "Sigma MCMC      :", sigma
)

print(
    "Random seed     :", random_seed
)

print("\nPersiapan selesai.")

Menyiapkan Subset Simulation...

Jumlah variabel : 7
Jumlah sampel   : 20000
p0              : 0.1
Sigma MCMC      : 1.0
Random seed     : 20261002

Persiapan selesai.


In [13]:
# ============================================================
# CELL 3H
# SUBSET SIMULATION
# ============================================================

print("=" * 70)
print("MEMULAI SUBSET SIMULATION")
print("=" * 70)


# ------------------------------------------------------------
# Parameter
# ------------------------------------------------------------

n_dim = len(X_complete.columns)

n_seed = int(
    round(N * p0)
)

chain_length = int(
    round(1 / p0)
)


# ------------------------------------------------------------
# LEVEL 0
# ------------------------------------------------------------

Y = rng.normal(
    size=(N, n_dim)
)

G = np.array([
    performance_function(y)
    for y in Y
])


levels = []

probability_factor = 1.0

level = 0


print("\nLEVEL 0")
print(
    "Jumlah sampel :",
    N
)


# ============================================================
# LOOP SUBSET SIMULATION
# ============================================================

while True:

    # --------------------------------------------------------
    # Urutkan berdasarkan nilai G terkecil
    # --------------------------------------------------------

    order = np.argsort(
        G,
        kind="stable"
    )

    G_sorted = G[
        order
    ]


    # --------------------------------------------------------
    # Cek apakah subset 10% sudah mencapai rare event
    # --------------------------------------------------------

    if G_sorted[
        n_seed - 1
    ] <= 0:

        final_count = int(
            np.sum(G <= 0)
        )

        final_conditional_probability = (
            final_count / N
        )

        final_probability = (
            probability_factor
            * final_conditional_probability
        )


        levels.append({

            "Level": level,

            "Threshold_G": 0.0,

            "Batas_Latitude": 5.0,

            "Jumlah_Event": final_count,

            "Probabilitas_Kondisional":
                final_conditional_probability,

            "Probabilitas_Kumulatif":
                final_probability,

            "Acceptance_Rate":
                np.nan,

            "Status":
                "FINAL"

        })


        # ====================================================
        # SIMPAN POPULASI FINAL
        # ====================================================

        # Seluruh 20.000 sampel pada level terakhir
        Y_level_final = Y.copy()

        # Nilai performance function seluruh sampel
        G_level_final = G.copy()

        # Mask sampel yang benar-benar memenuhi rare event
        mask_rare_final = (
            G_level_final <= 0
        )

        # Sampel latent Gaussian yang memenuhi |latitude| <= 5°
        Y_rare_final = Y_level_final[
            mask_rare_final
        ].copy()

        print("\nSampel final berhasil disimpan.")
        print(
            "Jumlah seluruh sampel final :",
            len(Y_level_final)
        )
        print(
            "Jumlah sampel rare event    :",
            len(Y_rare_final)
        )

        break


    # --------------------------------------------------------
    # Threshold subset
    # --------------------------------------------------------

    threshold = float(
        G_sorted[
            n_seed - 1
        ]
    )


    latitude_boundary = (
        5.0 + threshold
    )


    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    seeds = Y[
        order[
            :n_seed
        ]
    ].copy()


    # --------------------------------------------------------
    # MCMC
    # --------------------------------------------------------

    new_Y = np.empty_like(
        Y
    )

    accepted = 0

    proposed = 0

    current_index = 0


    for seed_index in range(
        n_seed
    ):

        current = seeds[
            seed_index
        ].copy()


        # Simpan seed
        new_Y[
            current_index
        ] = current

        current_index += 1


        # ----------------------------------------------------
        # Bangkitkan rantai
        # ----------------------------------------------------

        for step in range(
            1,
            chain_length
        ):

            candidate = current.copy()


            # Pilih satu komponen secara acak
            j = int(
                rng.integers(
                    n_dim
                )
            )


            candidate[
                j
            ] += (
                sigma *
                rng.normal()
            )


            candidate_G = (
                performance_function(
                    candidate
                )
            )


            proposed += 1


            # ------------------------------------------------
            # Kandidat harus tetap berada dalam subset
            # ------------------------------------------------

            if candidate_G <= threshold:

                current_norm = np.dot(
                    current,
                    current
                )

                candidate_norm = np.dot(
                    candidate,
                    candidate
                )


                log_alpha = -0.5 * (
                    candidate_norm
                    - current_norm
                )


                random_value = max(
                    rng.random(),
                    1e-15
                )


                if (
                    log_alpha >= 0
                    or
                    math.log(
                        random_value
                    ) < log_alpha
                ):

                    current = candidate

                    accepted += 1


            # Simpan sampel
            new_Y[
                current_index
            ] = current

            current_index += 1


    # --------------------------------------------------------
    # Populasi level berikutnya
    # --------------------------------------------------------

    Y = new_Y


    G = np.array([
        performance_function(y)
        for y in Y
    ])


    # --------------------------------------------------------
    # Update probabilitas
    # --------------------------------------------------------

    probability_factor *= p0

    level += 1


    # --------------------------------------------------------
    # Acceptance rate
    # --------------------------------------------------------

    acceptance_rate = (
        accepted / proposed
        if proposed > 0
        else np.nan
    )


    # --------------------------------------------------------
    # Catat hasil
    # --------------------------------------------------------

    levels.append({

        "Level": level,

        "Threshold_G": threshold,

        "Batas_Latitude":
            latitude_boundary,

        "Jumlah_Event":
            int(np.sum(G <= 0)),

        "Probabilitas_Kondisional":
            p0,

        "Probabilitas_Kumulatif":
            np.nan,

        "Acceptance_Rate":
            acceptance_rate,

        "Status":
            "INTERMEDIATE"

    })


    # --------------------------------------------------------
    # Tampilkan progress
    # --------------------------------------------------------

    print(
        f"\nLevel {level}"
    )

    print(
        f"Threshold G      : {threshold:.4f}"
    )

    print(
        f"Batas latitude   : ±{latitude_boundary:.2f}°"
    )

    print(
        f"Event ekuatorial : {int(np.sum(G <= 0))}"
    )

    print(
        f"Acceptance rate  : {acceptance_rate:.4f}"
    )


# ============================================================
# HASIL AKHIR
# ============================================================

results_ss = pd.DataFrame(
    levels
)


print("\n" + "=" * 70)
print("HASIL AKHIR SUBSET SIMULATION")
print("=" * 70)

print(
    results_ss.to_string(
        index=False
    )
)


print("\nProbabilitas Rare Event:")

print(
    f"{final_probability:.8f}"
)

print(
    f"{final_probability * 100:.4f}%"
)


print("\nCELL 3H SELESAI.")

MEMULAI SUBSET SIMULATION

LEVEL 0
Jumlah sampel : 20000

Level 1
Threshold G      : 3.5000
Batas latitude   : ±8.50°
Event ekuatorial : 1037
Acceptance rate  : 0.6308

Level 2
Threshold G      : 0.8000
Batas latitude   : ±5.80°
Event ekuatorial : 10297
Acceptance rate  : 0.6052

Sampel final berhasil disimpan.
Jumlah seluruh sampel final : 20000
Jumlah sampel rare event    : 10297

HASIL AKHIR SUBSET SIMULATION
 Level  Threshold_G  Batas_Latitude  Jumlah_Event  Probabilitas_Kondisional  Probabilitas_Kumulatif  Acceptance_Rate       Status
     1          3.5             8.5          1037                   0.10000                     NaN         0.630778 INTERMEDIATE
     2          0.8             5.8         10297                   0.10000                     NaN         0.605222 INTERMEDIATE
     2          0.0             5.0         10297                   0.51485                0.005149              NaN        FINAL

Probabilitas Rare Event:
0.00514850
0.5149%

CELL 3H SELESAI.


In [14]:
# ====================================================
# SIMPAN POPULASI FINAL
# ====================================================

Y_level_final = Y.copy()
G_level_final = G.copy()

mask_rare_final = (
    G_level_final <= 0
)

Y_rare_final = Y_level_final[
    mask_rare_final
].copy()

print("\nSampel final berhasil disimpan.")
print(
    "Jumlah seluruh sampel final :",
    len(Y_level_final)
)
print(
    "Jumlah sampel rare event    :",
    len(Y_rare_final)
)

# ----------------------------------------------------
# SIMPAN KE FILE JUGA
# Supaya tidak hilang kalau kernel mati lagi
# ----------------------------------------------------

np.savez_compressed(
    "subset_simulation_final.npz",
    Y_level_final=Y_level_final,
    G_level_final=G_level_final,
    Y_rare_final=Y_rare_final,
    L=L
)

X_complete.to_csv(
    "X_complete_model.csv",
    index=False
)

print(
    "\n✓ Backup subset_simulation_final.npz dibuat"
)
print(
    "✓ Backup X_complete_model.csv dibuat"
)


Sampel final berhasil disimpan.
Jumlah seluruh sampel final : 20000
Jumlah sampel rare event    : 10297

✓ Backup subset_simulation_final.npz dibuat
✓ Backup X_complete_model.csv dibuat


In [15]:
# ============================================================
# CELL 3I-A
# PEMERIKSAAN SAMPEL FINAL SUBSET SIMULATION
# ============================================================

print("=" * 75)
print("PEMERIKSAAN POPULASI FINAL SUBSET SIMULATION")
print("=" * 75)

print(
    "Shape Y_level_final :",
    Y_level_final.shape
)

print(
    "Shape G_level_final :",
    G_level_final.shape
)

print(
    "Shape Y_rare_final  :",
    Y_rare_final.shape
)

print(
    "\nJumlah G <= 0       :",
    np.sum(
        G_level_final <= 0
    )
)

print(
    "Proporsi final      :",
    np.mean(
        G_level_final <= 0
    )
)

print(
    "Probabilitas SS     :",
    p0 * p0 *
    np.mean(
        G_level_final <= 0
    )
)

print(
    "Probabilitas SS (%) :",
    100 *
    p0 * p0 *
    np.mean(
        G_level_final <= 0
    )
)

assert (
    Y_level_final.shape[0]
    == N
), "Jumlah sampel final bukan N"

assert (
    len(Y_rare_final)
    == final_count
), "Jumlah rare event tidak sesuai final_count"

print("\n✓ Pemeriksaan selesai.")

PEMERIKSAAN POPULASI FINAL SUBSET SIMULATION
Shape Y_level_final : (20000, 7)
Shape G_level_final : (20000,)
Shape Y_rare_final  : (10297, 7)

Jumlah G <= 0       : 10297
Proporsi final      : 0.51485
Probabilitas SS     : 0.005148500000000001
Probabilitas SS (%) : 0.51485

✓ Pemeriksaan selesai.


In [16]:
# ============================================================
# CELL 3I-B
# MELIHAT DEFINISI PERFORMANCE FUNCTION
# ============================================================

import inspect

print("=" * 75)
print("DEFINISI performance_function")
print("=" * 75)

try:
    source_pf = inspect.getsource(
        performance_function
    )

    print(source_pf)

except Exception as e:

    print(
        "inspect.getsource tidak berhasil:",
        e
    )

    print(
        "\nNama objek/fungsi yang digunakan "
        "oleh performance_function:"
    )

    print(
        performance_function.__code__.co_names
    )

print("\n" + "=" * 75)

DEFINISI performance_function
def performance_function(
    y
):

    # y = independent standard normal space

    z = L.dot(y)

    latitude = latitude_from_z(
        z[0]
    )

    return abs(latitude) - 5.0




In [17]:
# ============================================================
# CELL 3I-C LITE
# TRANSFORMASI SAMPEL RARE EVENT
# VERSI RINGAN - TANPA SCIPY
# ============================================================

import numpy as np
import pandas as pd
import math
from statistics import NormalDist

print("=" * 80)
print("TRANSFORMASI RARE EVENT - MODE LITE")
print("=" * 80)


# ============================================================
# 1. LOAD BACKUP
# ============================================================

print("\n[1/7] Membaca backup...")

backup = np.load(
    "subset_simulation_final.npz",
    allow_pickle=False
)

Y_rare = backup["Y_rare_final"]
L_safe = backup["L"]

X_obs = pd.read_csv(
    "X_complete_model.csv"
)

print("    Y_rare shape :", Y_rare.shape)
print("    L shape      :", L_safe.shape)
print("    X_obs shape  :", X_obs.shape)


# ============================================================
# 2. VALIDASI UKURAN
# ============================================================

print("\n[2/7] Memeriksa ukuran...")

if Y_rare.ndim != 2:
    raise ValueError("Y_rare bukan array 2 dimensi.")

if Y_rare.shape[1] != 7:
    raise ValueError(
        f"Jumlah variabel = {Y_rare.shape[1]}, seharusnya 7."
    )

if L_safe.shape != (7, 7):
    raise ValueError(
        f"Ukuran L = {L_safe.shape}, seharusnya (7, 7)."
    )

print("    Pemeriksaan ukuran OK")


# ============================================================
# 3. UJI SATU SAMPEL DULU
# ============================================================

print("\n[3/7] Menguji transformasi satu sampel...")

y_test = Y_rare[0]

# Sama seperti performance_function:
z_test = L_safe.dot(y_test)

print("    y pertama :", y_test)
print("    z pertama :", z_test)

print("    Transformasi satu sampel OK")


# ============================================================
# 4. FUNGSI NORMAL CDF TANPA SCIPY
# ============================================================

normal_std = NormalDist(
    mu=0.0,
    sigma=1.0
)

def normal_cdf_safe(x):
    return normal_std.cdf(
        float(x)
    )


# ============================================================
# 5. FUNGSI INVERSE EMPIRICAL QUANTILE RINGAN
# ============================================================

def inverse_empirical_light(
    sorted_values,
    u
):
    """
    Interpolasi linear kuantil empiris.

    sorted_values : data observasi yang sudah diurutkan
    u             : probabilitas 0-1
    """

    n = len(sorted_values)

    if n == 1:
        return float(
            sorted_values[0]
        )

    u = min(
        max(float(u), 0.0),
        1.0
    )

    position = (
        u * (n - 1)
    )

    lower = int(
        math.floor(position)
    )

    upper = int(
        math.ceil(position)
    )

    if lower == upper:
        return float(
            sorted_values[lower]
        )

    weight = (
        position - lower
    )

    return float(
        sorted_values[lower]
        +
        weight
        *
        (
            sorted_values[upper]
            -
            sorted_values[lower]
        )
    )


# ============================================================
# 6. SIAPKAN DATA OBSERVASI
# ============================================================

print("\n[4/7] Menyiapkan marginal empiris...")

columns = list(
    X_obs.columns
)

print("\nUrutan kolom model:")

for i, col in enumerate(columns):
    print(
        f"    {i}: {col}"
    )


# hanya meteorologi, latitude tidak perlu ditransformasi
meteo_columns = columns[1:]

sorted_obs = {}

for col in meteo_columns:

    values = pd.to_numeric(
        X_obs[col],
        errors="coerce"
    ).dropna().to_numpy(
        dtype=float
    )

    values.sort()

    sorted_obs[col] = values

    print(
        f"    {col}: {len(values)} data"
    )


# ============================================================
# 7. TRANSFORMASI SECARA BERTAHAP
# ============================================================

print("\n[5/7] Transformasi 10.297 sampel...")
print("      Mohon tunggu, dilakukan secara bertahap.")

n = Y_rare.shape[0]

hasil = {
    col: np.empty(
        n,
        dtype=float
    )
    for col in meteo_columns
}


# Tidak menyimpan seluruh Z dan U.
# Satu sampel diproses lalu dibuang.
for i in range(n):

    # sama persis dengan performance_function
    z = L_safe.dot(
        Y_rare[i]
    )

    # kolom 0 = latitude
    # kolom 1-6 = meteorologi
    for j, col in enumerate(
        meteo_columns,
        start=1
    ):

        u = normal_cdf_safe(
            z[j]
        )

        hasil[col][i] = (
            inverse_empirical_light(
                sorted_obs[col],
                u
            )
        )

    # Progress setiap 1000 sampel
    if (
        (i + 1) % 1000 == 0
        or
        i == n - 1
    ):
        print(
            f"      selesai {i + 1:,}/{n:,}"
        )


# ============================================================
# DATAFRAME HASIL
# ============================================================

print("\n[6/7] Membentuk DataFrame...")

X_rare_meteo = pd.DataFrame(
    hasil
)


print(
    "    Shape:",
    X_rare_meteo.shape
)

print("\n5 sampel pertama:")

print(
    X_rare_meteo.head()
)


# ============================================================
# SIMPAN LANGSUNG
# ============================================================

print("\n[7/7] Menyimpan hasil...")

X_rare_meteo.to_csv(
    "rare_event_meteorologi_final.csv",
    index=False
)

print(
    "    File: rare_event_meteorologi_final.csv"
)

print("\n" + "=" * 80)
print("CELL 3I-C LITE SELESAI")
print("=" * 80)

TRANSFORMASI RARE EVENT - MODE LITE

[1/7] Membaca backup...
    Y_rare shape : (10297, 7)
    L shape      : (7, 7)
    X_obs shape  : (385, 7)

[2/7] Memeriksa ukuran...
    Pemeriksaan ukuran OK

[3/7] Menguji transformasi satu sampel...
    y pertama : [-1.33525328 -1.16806295  0.83375837 -0.00441133 -0.41405862  0.1179918
 -0.76877521]
    z pertama : [-1.33525328 -1.3114687   0.9584911  -0.43148382 -0.62879261  0.18169361
 -0.48228601]
    Transformasi satu sampel OK

[4/7] Menyiapkan marginal empiris...

Urutan kolom model:
    0: Latitude
    1: SST
    2: Total_Precipitation
    3: Vertical_Velocity
    4: Relative_Vorticity
    5: Wind
    6: Pressure
    SST: 385 data
    Total_Precipitation: 385 data
    Vertical_Velocity: 385 data
    Relative_Vorticity: 385 data
    Wind: 385 data
    Pressure: 385 data

[5/7] Transformasi 10.297 sampel...
      Mohon tunggu, dilakukan secara bertahap.
      selesai 1,000/10,297
      selesai 2,000/10,297
      selesai 3,000/10,297
      

In [18]:
# ============================================================
# CELL 3I-D
# STATISTIK TABEL 3.13
# ============================================================

import pandas as pd
import numpy as np

df_rare = pd.read_csv(
    "rare_event_meteorologi_final.csv"
)

print(
    "Jumlah sampel:",
    len(df_rare)
)

print(
    "Kolom:",
    df_rare.columns.tolist()
)


# ------------------------------------------------------------
# Deteksi urutan enam parameter
# ------------------------------------------------------------

cols = list(
    df_rare.columns
)

if len(cols) != 6:
    raise ValueError(
        "Seharusnya terdapat 6 parameter meteorologis."
    )


# ------------------------------------------------------------
# Statistik
# ------------------------------------------------------------

records = []

for col in cols:

    x = pd.to_numeric(
        df_rare[col],
        errors="coerce"
    ).dropna()

    records.append({

        "Parameter":
            col,

        "Median":
            x.quantile(0.50),

        "Q1":
            x.quantile(0.25),

        "Q3":
            x.quantile(0.75),

        "P5":
            x.quantile(0.05),

        "P95":
            x.quantile(0.95),

        "Minimum":
            x.min(),

        "Maximum":
            x.max()
    })


stats_rare = pd.DataFrame(
    records
)

print("\nSTATISTIK RARE EVENT:")

print(
    stats_rare.round(5).to_string(
        index=False
    )
)

Jumlah sampel: 10297
Kolom: ['SST', 'Total_Precipitation', 'Vertical_Velocity', 'Relative_Vorticity', 'Wind', 'Pressure']

STATISTIK RARE EVENT:
          Parameter     Median         Q1         Q3        P5        P95   Minimum    Maximum
                SST   29.31478   28.96456   29.80598  28.37966   30.27318  27.04537   30.98495
Total_Precipitation    1.47790    0.54069    3.50392   0.07057    7.54303   0.00554   19.97625
  Vertical_Velocity   -0.20489   -0.39635   -0.01762  -1.02700    0.22048  -3.43078    0.78476
 Relative_Vorticity    8.82411    4.42283   13.42627 -22.04955   21.71335 -47.28790   49.88505
               Wind   45.00000   45.00000   45.00000  45.00000   55.00000  44.03119   92.48295
           Pressure 1004.00000 1002.00000 1004.00000 998.00000 1005.00000 987.57900 1008.80773
